# ASO → IDT order format

Turn an antisense oligonucleotide — **sequence** + per-position **sugar chemistry** + **phosphorothioate (PS) pattern** — into an IDT-style order string, using the shared `tauso.common.modifications.to_idt_notation`.

- **Chemical (sugar) pattern** — one letter per base, so its length equals the sequence length: `M` = 2′-MOE · `L` = LNA · `d` = DNA
- **PS pattern** — one character per inter-nucleotide linkage, so its length equals *sequence length − 1*: `*` = phosphorothioate · anything else = phosphodiester. Omit it for an all-PS backbone.
- **5-methyl-cytosine** — cytosines are 5-methyl by default: the DNA-gap C becomes `/iMe-dC/`, and the 2′-MOE / LNA wing C is already the 5-methyl form (`/i2MOErC/`, `+C`). Pass `modification_string="no_methyl_cytosine"` for plain dC.

`to_idt_notation` validates both lengths (chemical pattern = *L*, PS pattern = *L − 1*) and raises `ValueError` on a mismatch or an unsupported base/sugar. cEt is not an IDT product, so it raises rather than rendering.

> ⚠️ Confirm the emitted modification codes against IDT's current catalogue before ordering.

In [ ]:
from tauso.common.modifications import to_idt_notation

def standard_gapmer(sequence, chemistry="MOE"):
    """Shortcut chemical pattern for a standard gapmer: modified wings + DNA gap, with the
    reasonable default wing size per chemistry (2'-MOE -> 5-nt wings, e.g. a 20-mer is 5-10-5;
    LNA -> 3-nt wings, e.g. a 16-mer is 3-10-3). `chemistry` defaults to 2'-MOE."""
    wing, code = {"MOE": (5, "M"), "LNA": (3, "L")}[chemistry]
    return code * wing + "d" * (len(sequence) - 2 * wing) + code * wing

## One ASO

Type a **sequence** and a **chemical pattern** (one sugar letter per base), then run. For a standard 2′-MOE or LNA gapmer you can skip typing the pattern and use `standard_gapmer(sequence)` / `standard_gapmer(sequence, "LNA")` instead. Cytosines are 5-methyl by default (the canonical gapmer convention); pass `modification_string="no_methyl_cytosine"` for plain dC.

In [ ]:
sequence         = "GCAGTTATATTAGGTTCTCG"       # bases, 5'->3'
chemical_pattern = "MMMMMddddddddddMMMMM"        # one sugar per base: M=2'MOE  L=LNA  d=DNA
ps_pattern       = "*" * (len(sequence) - 1)     # optional; all-PS by default

# cytosines are 5-methyl by default (gap C -> /iMe-dC/, wing C already 5-methyl via /i2MOErC/):
print(to_idt_notation(sequence, chemical_pattern, ps_pattern))

# shortcut — a standard 2'-MOE gapmer gives the same pattern without typing it out:
print(to_idt_notation(sequence, standard_gapmer(sequence)))

# opt out of 5-methyl-cytosine (plain dC in the DNA gap):
print(to_idt_notation(sequence, standard_gapmer(sequence), modification_string="no_methyl_cytosine"))

## MALAT1 top-3 — TAUSO and OligoAI

Top 3 predicted ASOs from each model for MALAT1 (A549 / Lipofection / 100 nM, 2′-MOE 5-10-5 gapmer, full PS, 5-methyl-cytosine on every C).

In [ ]:
import pandas as pd

# (model, rank, target_start, sequence 5'->3'); cytosines are 5-methyl by default
specs = [
    ("TAUSO",   1, 4637, "GCCACTTCCTTTGCTCTGCA"),
    ("TAUSO",   2, 5346, "TCTCATTTATTTCGGCTTCT"),
    ("TAUSO",   3, 5762, "CCTTAGTTGGCATCAAGGCA"),
    ("OligoAI", 1, 8664, "GCAGTTATATTAGGTTCTCG"),
    ("OligoAI", 2, 6984, "GGAATGTTTCTTGTCACTGA"),
    ("OligoAI", 3, 8729, "GTTAAGTTTTCAGCAGTAGG"),
]

pd.DataFrame(
    [{"model": m, "rank": r, "target_start": t, "sequence": s,
      "idt_order": to_idt_notation(s, standard_gapmer(s))}
     for m, r, t, s in specs]
)